This notebook populates a neo4j Graph Database with Pokemon information fetched from the pokeapi API (up to date until Gen 9), and allows for questions regarding the database to gpt-4o.

In [1]:
import requests
import json
from neo4j import GraphDatabase
from pathlib import Path
import os
from dotenv import load_dotenv
import pandas as pd
from tqdm import tqdm
import openai
import time

In [2]:
dotenv_path = Path('~/.env').expanduser()
load_dotenv(dotenv_path=dotenv_path)  

# Get variables
POKEDEX_URI = os.getenv('POKEDEX_URI')
POKEDEX_USER = os.getenv('POKEDEX_USER')
POKEDEX_PASSWORD = os.getenv('POKEDEX_PASSWORD')
#database_name = os.getenv('DATABASE_NAME')

driver = GraphDatabase.driver(POKEDEX_URI, auth=(POKEDEX_USER, POKEDEX_PASSWORD))
driver.verify_connectivity()

------------------------------------------ **KG population** ------------------------------------------ 

In [3]:
# fetch pokemon data from given pokemon API
def fetch_pokemon_data(url):
    response = requests.get(url)
    return response.json()

def fetch_ability_description(ability_name):
    url = f"https://pokeapi.co/api/v2/ability/{ability_name.lower()}" # for abilities, API is different from base URL
    response = requests.get(url)
    data = response.json()
    for entry in data["effect_entries"]:
        if entry["language"]["name"] == "en":
            return entry["effect"]
    return None

base_url = "https://pokeapi.co/api/v2/pokemon" #base url

data = fetch_pokemon_data(base_url)

pokemon_list = []

# iterate through the pages and get data for each pokemon
while data:
    for pokemon in data['results']:
        pokemon_data = fetch_pokemon_data(pokemon['url'])

        time.sleep(0.5)

        species_data = fetch_pokemon_data(pokemon_data["species"]['url'])

        # extract relevant information
        name = pokemon_data["name"]
        types = [t["type"]["name"] for t in pokemon_data["types"]]
        generation = species_data["generation"]["name"]

        abilities = []
        ability_descriptions = []
        for ability_info in pokemon_data["abilities"]:
            ability = ability_info["ability"]["name"]
            desc = fetch_ability_description(ability)
            abilities.append(ability)
            ability_descriptions.append(desc)

        # append to list
        
        pokemon_list.append({"name": name, "types": types, "abilities": abilities, "ability_descriptions": ability_descriptions, "generation": generation})

    # get the next page (if any)
    next_url = data['next']
    if next_url:
        data = fetch_pokemon_data(next_url)
    else:
        break

# create collective df
df = pd.DataFrame(pokemon_list)
df['name'] = df['name'].str.capitalize()

print(df.head()) 

         name            types                abilities  \
0   Bulbasaur  [grass, poison]  [overgrow, chlorophyll]   
1     Ivysaur  [grass, poison]  [overgrow, chlorophyll]   
2    Venusaur  [grass, poison]  [overgrow, chlorophyll]   
3  Charmander           [fire]     [blaze, solar-power]   
4  Charmeleon           [fire]     [blaze, solar-power]   

                                ability_descriptions    generation  
0  [When this Pokémon has 1/3 or less of its HP r...  generation-i  
1  [When this Pokémon has 1/3 or less of its HP r...  generation-i  
2  [When this Pokémon has 1/3 or less of its HP r...  generation-i  
3  [When this Pokémon has 1/3 or less of its HP r...  generation-i  
4  [When this Pokémon has 1/3 or less of its HP r...  generation-i  


In [4]:
df['evolves_from'] = None

# Loop through Pokémon and fetch evolution info
for idx, row in tqdm(df.iterrows(), total=len(df)):
    name = row['name'].lower()
    url = f"https://pokeapi.co/api/v2/pokemon-species/{name}"
    response = requests.get(url)

    if response.status_code == 200:
        species_data = response.json()
        evolves_from = species_data.get('evolves_from_species')
        if evolves_from:
            df.at[idx, 'evolves_from'] = evolves_from['name'].capitalize()

100%|██████████| 1302/1302 [03:13<00:00,  6.74it/s]


In [ ]:
# # clean up data!!
# df['types'] = df['types'].apply(lambda x: [t.strip().capitalize() for t in x.split(',')] if isinstance(x, str) else x)
# df['abilities'] = df['abilities'].apply(lambda x: [t.strip().capitalize() for t in x.split(',')] if isinstance(x, str) else x)

# def split_commas(list):
#     result = []
#     for t in list:
#         if isinstance(t, str) and ',' in t:
#             result.extend([s.strip().capitalize() for s in t.split(',')])
#         else:
#             result.append(t.strip().capitalize() if isinstance(t, str) else t)
#     return result

# df['types'] = df['types'].apply(split_commas)
# df['abilities'] = df['abilities'].apply(split_commas)

# cleaned_generations = []
# for generation in df["generation"]:
#     generation = generation.split(sep="-")
#     generation = generation[1]
#     cleaned_generations.append(generation)
# df["generation"] = cleaned_generations

# print(df.head())

         name            types                abilities  \
0   Bulbasaur  [Grass, Poison]  [Overgrow, Chlorophyll]   
1     Ivysaur  [Grass, Poison]  [Overgrow, Chlorophyll]   
2    Venusaur  [Grass, Poison]  [Overgrow, Chlorophyll]   
3  Charmander           [Fire]     [Blaze, Solar-power]   
4  Charmeleon           [Fire]     [Blaze, Solar-power]   

                                ability_descriptions generation  
0  This Pokémon's Speed is doubled during strong ...          i  
1  This Pokémon's Speed is doubled during strong ...          i  
2  This Pokémon's Speed is doubled during strong ...          i  
3  During strong sunlight, this Pokémon has 1.5× ...          i  
4  During strong sunlight, this Pokémon has 1.5× ...          i  


In [7]:
def write_data(tx,statement, params_dict):
    tx.run(statement, parameters=params_dict)

In [ ]:
statement_create_pokemon = """MERGE (p:Pokemon {name:$name})"""

with driver.session() as session:
    for index, row in df.iterrows():
        session.execute_write(write_data, 
                                params_dict = {
                                    'name':str(row['name'])
                                },
                                statement = statement_create_pokemon)  

In [ ]:
statement_create_types = """MERGE (t:Type {name:$type})"""

with driver.session() as session:
    for index, row in df.iterrows():
        for type_name in row['types']:  # type_name will be 'water', 'grass', etc.
            session.execute_write(
                write_data, 
                params_dict={
                    'type': str(type_name).capitalize()  # capitalize type names
                },
                statement=statement_create_types
            )

In [39]:
statement_create_abilities = """
MERGE (a:Ability {name:$ability})
SET a.description = $description
"""

with driver.session() as session:
    for index, row in df.iterrows():
        # iterate abilities and descriptions together
        for ability_name, ability_desc in zip(row['abilities'], row['ability_descriptions']):
            session.execute_write(
                write_data, 
                params_dict={
                    'ability': str(ability_name).capitalize(),
                    'description': str(ability_desc)
                },
                statement=statement_create_abilities
            )

In [6]:
statement_create_generation = """MERGE (g:Generation {number:$generation})"""

with driver.session() as session:
    for generation_number in df['generation']:  
        session.execute_write(
            write_data, 
            params_dict={
                'generation': str(generation_number) 
            },
                statement=statement_create_generation
        )

In [ ]:
statement_has_type = """MATCH (p:Pokemon {name: $name})
MATCH (t:Type {name:$type})
MERGE (p)-[:HAS_TYPE]->(t)
"""

with driver.session() as session:
    for index, row in df.iterrows():
        pokemon_name = row['name']
        for type_name in row['types']:
            session.execute_write(
                write_data,
                params_dict = {
                    'name': pokemon_name,
                    'type': type_name
                },
                statement = statement_has_type
            )

C:\Users\Eleftheria\AppData\Local\Temp\ipykernel_19308\3795341116.py:6: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:


In [ ]:
statement_has_ability = """MATCH (p:Pokemon {name: $name})
MATCH (a:Ability {name:$ability})
MERGE (p)-[:HAS_ABILITY]->(a)
"""

with driver.session() as session:
    for index, row in df.iterrows():
        pokemon_name = row['name']
        for ability_name in row['abilities']:
            session.execute_write(
                write_data,
                params_dict = {
                    'name': pokemon_name,
                    'ability': ability_name
                },
                statement = statement_has_ability
            )

In [13]:
statement_in_generation = """MATCH (p:Pokemon {name: $name})
MATCH (g:Generation {number:$generation})
MERGE (p)-[:INTRODUCED_IN]->(g)
"""

with driver.session() as session:
    for index, row in df.iterrows():
        pokemon_name = row['name']
        generation_number = row['generation']
        session.execute_write(
                write_data,
                params_dict = {
                    'name': pokemon_name,
                    'generation': generation_number
                },
                statement = statement_in_generation
        )

In [5]:
df

,name,types,abilities,ability_descriptions,generation,evolves_from
0,Bulbasaur,"[grass, poison]","[overgrow, chlorophyll]",[When this Pokémon has 1/3 or less of its HP r...,generation-i,None
1,Ivysaur,"[grass, poison]","[overgrow, chlorophyll]",[When this Pokémon has 1/3 or less of its HP r...,generation-i,Bulbasaur
2,Venusaur,"[grass, poison]","[overgrow, chlorophyll]",[When this Pokémon has 1/3 or less of its HP r...,generation-i,Ivysaur
3,Charmander,[fire],"[blaze, solar-power]",[When this Pokémon has 1/3 or less of its HP r...,generation-i,None
4,Charmeleon,[fire],"[blaze, solar-power]",[When this Pokémon has 1/3 or less of its HP r...,generation-i,Charmander
...,...,...,...,...,...,...
1297,Ogerpon-wellspring-mask,"[grass, water]",[water-absorb],"[Whenever a water-type move hits this Pokémon,...",generation-ix,None
1298,Ogerpon-hearthflame-mask,"[grass, fire]",[mold-breaker],[This Pokémon's moves completely ignore abilit...,generation-ix,None
1299,Ogerpon-cornerstone-mask,"[grass, rock]",[sturdy],"[When this Pokémon is at full HP, any hit that...",generation-ix,None
1300,Terapagos-terastal,[normal],[tera-shell],[The Pokémon’s shell contains the powers of ea...,generation-ix,None


In [11]:
statement_evolves_from = """MATCH (p1:Pokemon {name: $name1})
MATCH (p2:Pokemon {name: $name2})
MERGE (p2)<-[:EVOLVES_FROM]-(p1)
"""

with driver.session() as session:
    for index, row in df.iterrows():
        pokemon_name = row['name']
        evolves_from = row['evolves_from']
        if (evolves_from != 'None'):
            session.execute_write(
                    write_data,
                    params_dict = {
                        'name1': pokemon_name,
                        'name2': evolves_from
                    },
                    statement = statement_evolves_from
            )

---------------------------------------------------------- **LLM integration for question-answer rapport** ----------------------------------------------------------

In [13]:
def get_graph_schema():
    schema = {}
    with driver.session() as session:
        # Node properties
        node_props = session.run("""
            CALL db.schema.nodeTypeProperties()
            YIELD nodeLabels, propertyName, propertyTypes
            RETURN nodeLabels, propertyName, propertyTypes
        """).data()
        
        # Relationship properties
        rel_props = session.run("""
            CALL db.schema.relTypeProperties()
            YIELD relType, propertyName, propertyTypes
            RETURN relType, propertyName, propertyTypes
        """).data()
        
        schema["nodes"] = node_props
        schema["relationships"] = rel_props
    return schema


In [15]:
client = openai.OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def ask_gpt4o(prompt):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content

def run_cypher(query):
    with driver.session() as session:
        return session.run(query).data()

def question_to_cypher_and_answer(question):
    # prompt LLM to translate the question into a Cypher query

    schema = get_graph_schema()
    
    prompt = f"""
    You are a Cypher expert. Here is the schema of the graph database: 
    {schema}
    Given the following question, respond ONLY with the Cypher query. Take into account the schema you have to figure out what the node, relationship and property names are.
    Do NOT include markdown or code block formatting. Just return the plain Cypher.

    Question: "{question}"
    """
    cypher_query = ask_gpt4o(prompt)
    print("Generated Cypher Query:\n", cypher_query)

    results = run_cypher(cypher_query)
    print("Query Results:\n", results)

    interpretation_prompt = f"""
    Given the question: "{question}"
    And the results: {results}
    List the results. Do not interpret them and do not check whether their are factually correct.
    """
    explanation = ask_gpt4o(interpretation_prompt)
    return explanation


In [9]:
question = "What type is Empoleon?"
print(question_to_cypher_and_answer(question))

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Procedure.ProcedureWarning} {category: GENERIC} {title: The query used a procedure that generated a warning.} {description: The query used a procedure that generated a warning. (The field `propertyTypes` will change output format in the next major version.)} {position: line: 2, column: 13, offset: 13} for query: '\n            CALL db.schema.nodeTypeProperties()\n            YIELD nodeLabels, propertyName, propertyTypes\n            RETURN nodeLabels, propertyName, propertyTypes\n        '
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Procedure.ProcedureWarning} {category: GENERIC} {title: The query used a procedure that generated a warning.} {description: The query used a procedure that generated a warning. (The field `propertyTypes` will change output format in the next major version.)} {position: line: 2, column: 13, offset: 13} for query: '\n        

Generated Cypher Query:
 MATCH (p:Pokemon {name: 'Empoleon'})-[:HAS_TYPE]->(t:Type) RETURN t.name
Query Results:
 [{'t.name': 'Water'}, {'t.name': 'Steel'}]
Water, Steel


------------------------------ **for visualisation purposes** -------------------------------------

In [17]:
import tkinter as tk
from tkinter import scrolledtext

def question_to_cypher_and_answer(question):
    schema = get_graph_schema()
    
    prompt = f"""
    You are a Cypher expert. Here is the schema of the graph database: 
    {schema}
    Given the following question, respond ONLY with the Cypher query. Take into account the schema you have to figure out what the node, relationship and property names are.
    Do NOT include markdown or code block formatting. Just return the plain Cypher.

    Question: "{question}"
    """
    cypher_query = ask_gpt4o(prompt)
    print("Generated Cypher Query:\n", cypher_query)

    results = run_cypher(cypher_query)
    print("Query Results:\n", results)

    interpretation_prompt = f"""
    Given the question: "{question}"
    And the results: {results}
    List the results. Do not interpret them and do not check whether their are factually correct.
    """
    explanation = ask_gpt4o(interpretation_prompt)
    return explanation

def ask_question():
    user_input = entry.get()
    if not user_input.strip():
        return
    response = question_to_cypher_and_answer(user_input)
    output_box.config(state="normal")  # enable editing
    output_box.insert(tk.END, f"> {user_input}\n{response}\n\n")
    output_box.config(state="disabled")  # lock editing
    entry.delete(0, tk.END)

# main window
root = tk.Tk()
root.title("Pokédex")
root.configure(bg="#d62828")
root.iconbitmap("rotom.ico")

# input 
entry = tk.Entry(root, width=50, font=("Consolas", 12), relief="solid", bd=3)
entry.grid(row=0, column=0, padx=10, pady=10)

# ask button
ask_button = tk.Button(root, text="Ask Pokédex", command=ask_question, bg="#cfc539", fg="white", font=("Arial", 12, "bold"), relief="raised", bd=4)
ask_button.grid(row=0, column=1, padx=5, pady=10)

# output box
output_box = scrolledtext.ScrolledText(root, wrap=tk.WORD, width=60, height=20, font=("Consolas", 11))
output_box.grid(row=1, column=0, columnspan=2, padx=10, pady=10)
output_box.config(state="disabled")

root.mainloop()

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Procedure.ProcedureWarning} {category: GENERIC} {title: The query used a procedure that generated a warning.} {description: The query used a procedure that generated a warning. (The field `propertyTypes` will change output format in the next major version.)} {position: line: 2, column: 13, offset: 13} for query: '\n            CALL db.schema.nodeTypeProperties()\n            YIELD nodeLabels, propertyName, propertyTypes\n            RETURN nodeLabels, propertyName, propertyTypes\n        '
Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Procedure.ProcedureWarning} {category: GENERIC} {title: The query used a procedure that generated a warning.} {description: The query used a procedure that generated a warning. (The field `propertyTypes` will change output format in the next major version.)} {position: line: 2, column: 13, offset: 13} for query: '\n        

Generated Cypher Query:
 MATCH (p:Pokemon {name: 'Meowscarada'})-[:EVOLVES_FROM*]->(e:Pokemon)
RETURN e.name
Query Results:
 [{'e.name': 'Floragato'}, {'e.name': 'Sprigatito'}]
